In [33]:
import pandas as pd
import datetime
import os
import json
import altair as alt

import base64
from io import BytesIO
from PIL import Image


df = pd.read_csv("Gesamtdatensatz.csv")

image_paths = ["src/assets/clear-day.png","src/assets/clear-night.png", "src/assets/cloudy.png", "src/assets/fog.png", "src/assets/partly-cloudy-day.png", "src/assets/partly-cloudy-night.png", "src/assets/rain.png", "src/assets/snow.png"]
path = {}

for image_path in image_paths:
    # Schlüssel aus Dateiname extrahieren (ohne Ordner und ohne .png)
    key = os.path.splitext(os.path.basename(image_path))[0]
    # Pfad in das Dictionary einfügen
    path[key] = image_path



# Add columns to table
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['pedestrian_grey'] = df[['ltr_pedestrians_count', 'rtl_pedestrians_count']].min(axis=1)
df['pedestrian_diff'] = ((df['ltr_pedestrians_count'] - df['rtl_pedestrians_count'])**2)**0.5
df['weather_icon'] = df['weather_condition'].map(path)
df['max_val'] = (
    df.groupby(['date'])[['ltr_pedestrians_count', 'rtl_pedestrians_count']]
      .transform('max')      # max per column per date
      .max(axis=1)           # max across the two columns
)+20


# Additional filtering for Adult and Children Charts:
df['child_pedestrian_grey'] = df[['child_ltr_pedestrians_count', 'child_rtl_pedestrians_count']].min(axis=1)
df['child_pedestrian_diff_chi'] = ((df['child_ltr_pedestrians_count'] - df['child_rtl_pedestrians_count'])**2)**0.5
df['child_max_val'] = (
    df.groupby(['date'])[['child_ltr_pedestrians_count', 'child_rtl_pedestrians_count']]
      .transform('max')      # max per column per date
      .max(axis=1)           # max across the two columns
)+20

df['adult_pedestrian_grey'] = df[['adult_ltr_pedestrians_count', 'adult_rtl_pedestrians_count']].min(axis=1)
df['adult_pedestrian_diff_chi'] = ((df['adult_ltr_pedestrians_count'] - df['adult_rtl_pedestrians_count'])**2)**0.5
df['adult_max_val'] = (
    df.groupby(['date'])[['adult_ltr_pedestrians_count', 'adult_rtl_pedestrians_count']]
      .transform('max')      # max per column per date
      .max(axis=1)           # max across the two columns
)+20

df = df.rename(columns={'temperature': 'Temperatur'})

f_time_loc = df[(df['timestamp'].dt.date == datetime.date(2022, 1, 18)) & (df['location_name'] == 'Bahnhofstrasse (Mitte)')]
f_time_loc.to_json('time_loc_data.json', orient='records', indent=2)
f_time_loc.head(n=5)

,timestamp,location_id,location_name,ltr_label,rtl_label,weather_condition,Temperatur,pedestrians_count,unverified,collection_type,...,pedestrian_grey,pedestrian_diff,weather_icon,max_val,child_pedestrian_grey,child_pedestrian_diff_chi,child_max_val,adult_pedestrian_grey,adult_pedestrian_diff_chi,adult_max_val
10664,2022-01-18 00:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-0.75,10,False,measured,...,4,2.0,src/assets/partly-cloudy-night.png,1319,0,0.0,681,4,2.0,1308
10668,2022-01-18 01:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-1.27,16,False,measured,...,6,4.0,src/assets/partly-cloudy-night.png,1319,0,1.0,681,5,5.0,1308
10672,2022-01-18 02:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-1.47,5,False,measured,...,2,1.0,src/assets/partly-cloudy-night.png,1319,0,0.0,681,2,1.0,1308
10676,2022-01-18 03:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-1.84,10,False,measured,...,4,2.0,src/assets/partly-cloudy-night.png,1319,0,0.0,681,4,2.0,1308
10680,2022-01-18 04:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-2.43,96,False,measured,...,34,28.0,src/assets/partly-cloudy-night.png,1319,3,0.0,681,31,28.0,1308


In [34]:
# Chart All

#data = pd.read_csv("Gesamtdatensatz.csv")
#data = pd.read_json("time_loc_data.json")
data = 'time_loc_data.json'

# Initialize Basic Chart
base = alt.Chart(data).add_params().properties(width=370, height=24*25)

#Invisible Chart for Axis
inv = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None)), 
    ).mark_rect(
        opacity=0,
        height=0)

# Build Temperatur [°C]e chart for Background
temp_left = base.encode(
    alt.Y('hour:T').axis(None),
    alt.Color('Temperatur:Q')
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q', 
            title='Temperatur [°C]:', 
            format=".1f")]).mark_rect(
        height=25)

temp_right = base.encode(
    alt.Y('hour:O').axis(None),

    alt.Color('Temperatur:Q')
        #.bin(maxbins=10, extent=[-10,35])
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q',
            title='Temperatur [°C]:',
            format=".1f")]).mark_rect(
            height=25)

# Build Pedestrian-count chart
left = (base.transform_calculate(
        tooltip_title="datum.pedestrian_diff + ' Passanten mehr in Richtung ' + datum.rtl_label + ' als in Richtung ' + datum.ltr_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('ltr_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
                color='#6A11B3',
                height=20)
).interactive()                

right = (base.transform_calculate(
        tooltip_title="datum.pedestrian_diff + ' Passanten mehr in Richtung ' + datum.ltr_label + ' als in Richtung ' +datum.rtl_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('rtl_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color='#6A11B3', 
            height=20)
).interactive()
# Build Pedestrian-even count chart
left_g = (base.transform_calculate(
        tooltip_title="'Total Anzahl Passanten in Richtung ' + datum.rtl_label + ': ' + datum.ltr_pedestrians_count")
    .encode(
    alt.Y('hour:O').axis(None),
    alt.X('sum(pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom'))
        .sort('descending'),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color="#8950B8",
            height=20)
) .interactive()          

right_g = (base.transform_calculate(
        tooltip_title="'Total Anzahl Passanten in Richtung ' + datum.ltr_label + ': ' + datum.rtl_pedestrians_count")
    .encode(
        alt.Y('hour:O').axis(None),
        alt.X('sum(pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom')),
        tooltip=[alt.Tooltip('tooltip_title:N', title=' '),])
    .mark_bar(color='#8950B8', height=20)
).interactive()

# weather icon chart
weather = base.transform_aggregate(
    weather_icon='max(weather_icon)',
    groupby=['hour', 'location_id']
).encode(
    alt.Y('hour:O').axis(None),
    url='weather_icon:N'
).mark_image(width=25, height=25).properties(width=6)

# Build middle chart (legend)
middle = base.transform_aggregate(groupby=['hour', 'location_id']).encode(
    alt.Y('hour:O').axis(None),
    alt.Text('hour:O')).mark_text(
        fontSize=15,
        font='Bahnschrift').properties(width=20)

# Build hidden chart (vertical)
inv_r = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None, orient='bottom')).axis(None), 
    ).mark_rect(
        opacity=0,
        height=0).properties(width=30)


# Layer Charts
left_chart_a = alt.layer(left, left_g, inv)
right_chart_a = alt.layer(right, right_g, inv)
left_chart = alt.layer(temp_left, left_chart_a).resolve_scale(x='independent')
right_chart = alt.layer(temp_right, right_chart_a).resolve_scale(x='independent')

# Concatenate Charts
main_chart = alt.concat(left_chart, middle, right_chart, weather, inv_r, spacing=5).configure_view(
    stroke=None,
).configure_legend(direction='vertical', labelAlign='left', orient='left', gradientLength=24*25, gradientThickness=25, labelFont='Bahnschrift',
                   labelFontSize=12, titleOrient='left', titleFont='Bahnschrift', titleFontSize=17, titleFontWeight=300)

spec = main_chart.to_dict()

with open("chart_all.json", "w") as f:
    json.dump(spec, f, indent=2)

main_chart
#https://altair-viz.github.io/user_guide/marks/image.html


c:\Users\jonat\miniconda3\envs\3050WID\Lib\site-packages\IPython\core\interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.ConcatChart(...)

In [35]:
# Chart Children

#data = pd.read_csv("Gesamtdatensatz.csv")
#data = pd.read_json("time_loc_data.json")
data = 'time_loc_data.json'

# Initialize Basic Chart
base = alt.Chart(data).add_params().properties(width=370, height=24*25)

# Inivisible Chart for Axis
inv = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('child_max_val:Q',axis=alt.Axis(title=None)), 
    ).mark_rect(
        opacity=0,
        height=0)


# Build Temperatur [°C]e chart for Background
temp_left = base.encode(
    alt.Y('hour:T').axis(None), 
    alt.Color('Temperatur:Q')
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q', 
            title='Temperatur [°C]:', 
            format=".1f")]).mark_rect(
        height=25)

temp_right = base.encode(
    alt.Y('hour:O').axis(None),
    alt.Color('Temperatur:Q')
        #.bin(maxbins=10, extent=[-10,35])
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q',
            title='Temperatur [°C]:',
            format=".1f")]).mark_rect(
            height=25)

# Build Pedestrian-count chart
left = (base.transform_calculate(
        tooltip_title="datum.child_pedestrian_diff + ' Kinder mehr in Richtung ' + datum.rtl_label + ' als in Richtung ' + datum.ltr_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('child_ltr_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
                color='#6A11B3',
                height=20)
).interactive()                 

right = (base.transform_calculate(
        tooltip_title="datum.child_pedestrian_diff + ' Kinder mehr in Richtung ' + datum.ltr_label + ' als in Richtung ' +datum.rtl_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('child_rtl_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color='#6A11B3', 
            height=20)
).interactive() 
# Build Pedestrian-even count chart
left_g = (base.transform_calculate(
        tooltip_title="'Totale Anzahl Kinder in Richtung ' + datum.rtl_label + ': ' + datum.child_ltr_pedestrians_count")
    .encode(
    alt.Y('hour:O').axis(None),
    alt.X('sum(child_pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom'))
        .sort('descending'),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color="#8950B8",
            height=20)
).interactive()             

right_g = (base.transform_calculate(
        tooltip_title="'Totale Anzahl Kinder in Richtung ' + datum.ltr_label + ': ' + datum.child_rtl_pedestrians_count")
    .encode(
        alt.Y('hour:O').axis(None),
        alt.X('sum(child_pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom')),
        tooltip=[alt.Tooltip('tooltip_title:N', title=' '),])
    .mark_bar(color='#8950B8', height=20)
).interactive() 

# weather icon chart
weather = base.transform_aggregate(
    weather_icon='max(weather_icon)',
    groupby=['hour', 'location_id']
).encode(
    alt.Y('hour:O').axis(None),
    url='weather_icon:N'
).mark_image(width=25, height=25).properties(width=6)

# Build middle chart (legend)
middle = base.transform_aggregate(groupby=['hour', 'location_id']).encode(
    alt.Y('hour:O').axis(None),
    alt.Text('hour:O')).mark_text(
        fontSize=15,
        font='Bahnschrift').properties(width=20)

# Build hidden chart (vertical)
inv_r = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None, orient='bottom')).axis(None), 
    ).mark_rect(
        opacity=0,
        height=0).properties(width=30)


# Layer Charts
left_chart_a = alt.layer(left, left_g, inv)
right_chart_a = alt.layer(right, right_g, inv)
left_chart = alt.layer(temp_left, left_chart_a).resolve_scale(x='independent')
right_chart = alt.layer(temp_right, right_chart_a).resolve_scale(x='independent')

# Concatenate Charts
main_chart = alt.concat(left_chart, middle, right_chart, weather, inv_r, spacing = 5,).configure_view(
    stroke=None,
).configure_legend(direction='vertical', labelAlign='left', orient='left', gradientLength=24*25, gradientThickness=25, labelFont='Bahnschrift',
                   labelFontSize=12, titleOrient='left', titleFont='Bahnschrift', titleFontSize=17, titleFontWeight=300)

spec = main_chart.to_dict()

with open("chart_child.json", "w") as f:
    json.dump(spec, f, indent=2)

main_chart
#https://altair-viz.github.io/user_guide/marks/image.html


alt.ConcatChart(...)

In [36]:
# Chart Adult

#data = pd.read_csv("Gesamtdatensatz.csv")
#data = pd.read_json("time_loc_data.json")
data = 'time_loc_data.json'

# Initialize Basic Chart
base = alt.Chart(data).add_params().properties(width=370, height=24*25)

# Invisible Chart for Axis
inv = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('adult_max_val:Q',axis=alt.Axis(title=None)), 
    ).mark_rect(
        opacity=0,
        height=0)

# Build Temperature chart for Background
temp_left = base.encode(
    alt.Y('hour:T').axis(None), 
    alt.Color('Temperatur:Q')
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q', 
            title='Temperatur [°C]:', 
            format=".1f")]).mark_rect(
        height=25)

temp_right = base.encode(
    alt.Y('hour:O').axis(None),
    alt.Color('Temperatur:Q')
        #.bin(maxbins=10, extent=[-10,35])
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q',
            title='Temperatur [°C]:',
            format=".1f")]).mark_rect(
            height=25)

# Build Pedestrian-count chart
left = (base.transform_calculate(
        tooltip_title="datum.adult_pedestrian_diff + ' Erwachsene mehr in Richtung ' + datum.rtl_label + ' als in Richtung ' + datum.ltr_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('adult_ltr_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
                color='#6A11B3',
                height=20)
).interactive()                 

right = (base.transform_calculate(
        tooltip_title="datum.adult_pedestrian_diff + ' Erwachsene mehr in Richtung ' + datum.ltr_label + ' als in Richtung ' +datum.rtl_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('adult_rtl_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color='#6A11B3', 
            height=20)
).interactive() 
# Build Pedestrian-even count chart
left_g = (base.transform_calculate(
        tooltip_title="'Totale Anzahl Erwachsene in Richtung ' + datum.rtl_label + ': ' + datum.adult_ltr_pedestrians_count")
    .encode(
    alt.Y('hour:O').axis(None),
    alt.X('sum(adult_pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom'))
        .sort('descending'),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color="#8950B8",
            height=20)
).interactive()             

right_g = (base.transform_calculate(
        tooltip_title="'Totale Anzahl Erwachsene in Richtung ' + datum.ltr_label + ': ' + datum.adult_rtl_pedestrians_count")
    .encode(
        alt.Y('hour:O').axis(None),
        alt.X('sum(adult_pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom')),
        tooltip=[alt.Tooltip('tooltip_title:N', title=' '),])
    .mark_bar(color='#8950B8', height=20)
).interactive() 

# weather icon chart
weather = base.transform_aggregate(
    weather_icon='max(weather_icon)',
    groupby=['hour', 'location_id']
).encode(
    alt.Y('hour:O').axis(None),
    url='weather_icon:N'
).mark_image(width=25, height=25).properties(width=6)

# Build middle chart (legend)
middle = base.transform_aggregate(groupby=['hour', 'location_id']).encode(
    alt.Y('hour:O').axis(None),
    alt.Text('hour:O')).mark_text(
        fontSize=15,
        font='Bahnschrift').properties(width=20)

# Build hidden chart (vertical)
inv_r = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None, orient='bottom')).axis(None), 
    ).mark_rect(
        opacity=0,
        height=0).properties(width=30)


# Layer Charts
left_chart_a = alt.layer(left, left_g, inv)
right_chart_a = alt.layer(right, right_g, inv)
left_chart = alt.layer(temp_left, left_chart_a).resolve_scale(x='independent')
right_chart = alt.layer(temp_right, right_chart_a).resolve_scale(x='independent')

# Concatenate Charts
main_chart = alt.concat(left_chart, middle, right_chart, weather, inv_r, spacing = 5,).configure_view(
    stroke=None,
).configure_legend(direction='vertical', labelAlign='left', orient='left', gradientLength=24*25, gradientThickness=25, labelFont='Bahnschrift',
                   labelFontSize=12, titleOrient='left', titleFont='Bahnschrift', titleFontSize=17, titleFontWeight=300)

spec = main_chart.to_dict()

with open("chart_adult.json", "w") as f:
    json.dump(spec, f, indent=2)

main_chart
#https://altair-viz.github.io/user_guide/marks/image.html


alt.ConcatChart(...)

In [37]:
# Chart All

#data = pd.read_csv("Gesamtdatensatz.csv")
#data = pd.read_json("time_loc_data.json")
data = 'time_loc_data.json'

# Initialize Basic Chart
base = alt.Chart(data).add_params().properties(width=370, height=24*25)

#Invisible Chart for Axis
inv = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None)), 
    ).mark_rect(
        opacity=0,
        height=0)

# Build Temperatur [°C]e chart for Background
temp_left = base.encode(
    alt.Y('hour:T').axis(None),
    alt.Color('Temperatur:Q')
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q', 
            title='Temperatur [°C]:', 
            format=".1f")]).mark_rect(
        height=25)

temp_right = base.encode(
    alt.Y('hour:O').axis(None),

    alt.Color('Temperatur:Q')
        #.bin(maxbins=10, extent=[-10,35])
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('Temperatur:Q',
            title='Temperatur [°C]:',
            format=".1f")]).mark_rect(
            height=25)

# Build Pedestrian-count chart
left = (base.transform_calculate(
        tooltip_title="datum.pedestrian_diff + ' Passanten mehr in Richtung ' + datum.rtl_label + ' als in Richtung ' + datum.ltr_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('ltr_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
                color='#6A11B3',
                height=20)
).interactive()                

right = (base.transform_calculate(
        tooltip_title="datum.pedestrian_diff + ' Passanten mehr in Richtung ' + datum.ltr_label + ' als in Richtung ' +datum.rtl_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('rtl_pedestrians_count:Q', axis=alt.Axis(title=None, orient='bottom')),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color='#6A11B3', 
            height=20)
).interactive()
# Build Pedestrian-even count chart
left_g = (base.transform_calculate(
        tooltip_title="'Total Anzahl Passanten in Richtung ' + datum.rtl_label + ': ' + datum.ltr_pedestrians_count")
    .encode(
    alt.Y('hour:O').axis(None),
    alt.X('sum(pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom'))
        .sort('descending'),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color="#8950B8",
            height=20)
) .interactive()          

right_g = (base.transform_calculate(
        tooltip_title="'Total Anzahl Passanten in Richtung ' + datum.ltr_label + ': ' + datum.rtl_pedestrians_count")
    .encode(
        alt.Y('hour:O').axis(None),
        alt.X('sum(pedestrian_grey):Q', axis=alt.Axis(title=None, orient='bottom')),
        tooltip=[alt.Tooltip('tooltip_title:N', title=' '),])
    .mark_bar(color='#8950B8', height=20)
).interactive()

# weather icon chart
weather = base.transform_aggregate(
    weather_icon='max(weather_icon)',
    groupby=['hour', 'location_id']
).encode(
    alt.Y('hour:O').axis(None),
    url='weather_icon:N'
).mark_image(width=25, height=25).properties(width=6)

# Build middle chart (legend)
middle = base.transform_aggregate(groupby=['hour', 'location_id']).encode(
    alt.Y('hour:O').axis(None),
    alt.Text('hour:O')).mark_text(
        fontSize=15,
        font='Bahnschrift').properties(width=20)

# Build hidden chart (vertical)
inv_r = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None, orient='bottom')).axis(None), 
    ).mark_rect(
        opacity=0,
        height=0).properties(width=30)


# Layer Charts
left_chart_a = alt.layer(left, left_g, inv)
right_chart_a = alt.layer(right, right_g, inv)
left_chart = alt.layer(temp_left, left_chart_a).resolve_scale(x='independent')
right_chart = alt.layer(temp_right, right_chart_a).resolve_scale(x='independent')

# Concatenate Charts
main_chart = alt.concat(left_chart, middle, right_chart, weather, inv_r, spacing=5).configure_view(
    stroke=None,
).configure_legend(direction='vertical', labelAlign='left', orient='left', gradientLength=24*25, gradientThickness=25, labelFont='Bahnschrift',
                   labelFontSize=12, titleOrient='left', titleFont='Bahnschrift', titleFontSize=17, titleFontWeight=300)

#spec = main_chart.to_dict()

#with open("chart_all.json", "w") as f:
    #json.dump(spec, f, indent=2)

main_chart
#https://altair-viz.github.io/user_guide/marks/image.html


alt.ConcatChart(...)